In [ ]:
# IMPORTS Y CONFIGURACIÓN

import requests  #cliente HTTP para hacer GET a paginas web
from bs4 import BeautifulSoup #parser HTML para navegar el DOM
import pandas as pd #DataFrames para consultas/outputs tabulares
import time,random #Pausas y aleatoriedad 
import re #expresiones regulares para extraer patrones
from datetime import datetime #timestamp de cuando se scrapeo
import sqlite3 #base de datos

# Configuración
URL_BASE = "https://books.toscrape.com/" #URL semilla de donde parte el rastreador 
ENCABEZADOS = { #header para parecer navegador real(evita bloqueos basicos)
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

def pausa_aleatoria(): #funcion para que el servidor no piense que somos un robot
    time.sleep(random.uniform(1, 2))

print("🕵️‍♂️ Iniciando scrapeo COMPLETO...")

🕵️‍♂️ Iniciando scrapeo COMPLETO...


In [ ]:
# OBTENER TODAS LAS CATEGORÍAS

def obtener_todas_categorias():
    print("🔍 Buscando todas las categorías...")
    respuesta = requests.get(URL_BASE, headers=ENCABEZADOS) #descarga el HTML de portada con headers personalizados
    sopa = BeautifulSoup(respuesta.content, 'html.parser') #convierte HTML en arbol navegable 
    
    categorias = []
    barra_lateral = sopa.find('ul', class_='nav-list').find('ul') #ubica el menu de categorias: <ul.nav-list>, luego su <ul> interno
    
    for enlace in barra_lateral.find_all('a'): #itera todos los <a>(una categoria por enlace)
        nombre_categoria = enlace.text.strip() #limpia el texto visible del link(nombre de la categoria)
        url_categoria = URL_BASE + enlace['href'] #construye URL de la categoria(asume href relativo)
        
        categorias.append({ #aculuma resultados en una lista de diccionarios
            'nombre': nombre_categoria,
            'url': url_categoria
        })
        print(f"📍 {nombre_categoria}")
    
    print(f"✅ Encontradas {len(categorias)} categorías")
    return categorias #devuelve todas las categorias encontradas

# Ejecutar
todas_categorias = obtener_todas_categorias() # ejecuta el discovery y guarda el resultado

🔍 Buscando todas las categorías...
📍 Travel
📍 Mystery
📍 Historical Fiction
📍 Sequential Art
📍 Classics
📍 Philosophy
📍 Romance
📍 Womens Fiction
📍 Fiction
📍 Childrens
📍 Religion
📍 Nonfiction
📍 Music
📍 Default
📍 Science Fiction
📍 Sports and Games
📍 Add a comment
📍 Fantasy
📍 New Adult
📍 Young Adult
📍 Science
📍 Poetry
📍 Paranormal
📍 Art
📍 Psychology
📍 Autobiography
📍 Parenting
📍 Adult Fiction
📍 Humor
📍 Horror
📍 History
📍 Food and Drink
📍 Christian Fiction
📍 Business
📍 Biography
📍 Thriller
📍 Contemporary
📍 Spirituality
📍 Academic
📍 Self Help
📍 Historical
📍 Christian
📍 Suspense
📍 Short Stories
📍 Novels
📍 Health
📍 Politics
📍 Cultural
📍 Erotica
📍 Crime
✅ Encontradas 50 categorías


In [ ]:
# SCRAPEAR CATEGORÍAS COMPLETAS

def scrapear_categoria(url_categoria, nombre_categoria): #funcion que recorre todas las paginas de una categoria
    print(f"🎯 Scrapeando: {nombre_categoria}")
    
    todos_libros = []
    url_actual = url_categoria #estado del rastreador(crawler) para esa categoria
    numero_pagina = 1
    
    while url_actual:
        try:
            print(f"   📄 Página {numero_pagina}")
            respuesta = requests.get(url_actual, headers=ENCABEZADOS) #descarga cada pagina de la categoria 
            
            if respuesta.status_code != 200: #termina si hubo error HTTP
                break
                
            sopa = BeautifulSoup(respuesta.content, 'html.parser') #lista de fichas de libros en la pagina
            libros = sopa.find_all('article', class_='product_pod')
            
            if not libros:
                break
            
            for libro in libros:
                try:
                    titulo = libro.h3.a['title'] #extrae el titulo desde el <a> dentro del  <h3>
                    precio_texto = libro.find('p', class_='price_color').text 
                    precio = float(precio_texto.replace('£', '')) #parsea precio a float
                    
                    # Calificación
                    clase_rating = libro.p['class'][1] 
                    mapeo_rating = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
                    rating = mapeo_rating.get(clase_rating, 0) #decodifica estrellas
                    
                    # URL del libro
                    url_libro_relativa = libro.h3.a['href'] #toma el href de la ficha del libro
                    if url_libro_relativa.startswith('../../../'): #corrige rutas relativas complejas para formar url_libro_completa
                        url_libro_completa = URL_BASE + 'catalogue/' + url_libro_relativa.replace('../../../', '')
                    else:
                        url_libro_completa = URL_BASE + 'catalogue/' + url_libro_relativa
                    
                    datos_libro = {
                        'titulo': titulo,
                        'precio': precio,
                        'rating': rating,
                        'categoria': nombre_categoria,
                        'url': url_libro_completa
                    }
                    todos_libros.append(datos_libro) #guarda datos basicos del libro(sin detalles aun)
                    
                except Exception as error_libro:
                    continue
            
            # Verificar siguiente página
            boton_siguiente = sopa.find('li', class_='next') #detecta si hay siguiente pagina
            if boton_siguiente:
                # Paginación: si existe <li class="next"> toma su href y construye la URL de la siguiente página
                # Maneja distintos casos de URL (con/ sin 'catalogue/', index.html, page-X) para formar correctamente la ruta
                # Avanza el contador de página y aplica una pausa aleatoria (rate limiting); si no hay "next", termina el bucle

                enlace_siguiente = boton_siguiente.a['href']
                
                if 'catalogue/' in url_actual:
                    if '/page-' in url_actual:
                        base_url = '/'.join(url_actual.split('/')[:-1]) + '/'
                        url_actual = base_url + enlace_siguiente
                    else:
                        if url_actual.endswith('/'):
                            url_actual = url_actual + enlace_siguiente
                        else:
                            url_actual = url_actual.replace('index.html', enlace_siguiente)
                else:
                    if url_actual.endswith('index.html'):
                        url_actual = url_actual.replace('index.html', enlace_siguiente)
                    elif url_actual.endswith('/'):
                        url_actual = url_actual + enlace_siguiente
                    else:
                        url_actual = url_actual + '/' + enlace_siguiente
                
                numero_pagina += 1
                pausa_aleatoria()
            else:
                url_actual = None
                
        except Exception as error_pagina:
            print(f"Error:{error_pagina}")
            break
    
    print(f"✅ {nombre_categoria}: {len(todos_libros)} libros")
    return todos_libros

In [ ]:
# SCRAPEAR DETALLES DE CADA LIBRO

def scrapear_detalles_libro(url_libro):
    try:
        respuesta = requests.get(url_libro, headers=ENCABEZADOS)
        sopa = BeautifulSoup(respuesta.content, 'html.parser')
        
        # Información básica
        titulo = sopa.find('h1').text
        precio = float(sopa.find('p', class_='price_color').text.replace('£', ''))
        
        # Stock
        texto_stock = sopa.find('p', class_='instock availability').text
        coincidencia_stock = re.search(r'\((\d+) available\)', texto_stock)
        stock = int(coincidencia_stock.group(1)) if coincidencia_stock else 0
        
        # Rating
        clase_rating = sopa.find('p', class_='star-rating')['class'][1]
        mapeo_rating = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
        rating = mapeo_rating.get(clase_rating, 0)
        
        # Descripción
        meta_descripcion = sopa.find('meta', attrs={'name': 'description'})
        descripcion = meta_descripcion['content'].strip() if meta_descripcion else ""
        
        # Información de la tabla
        upc = sopa.find('th', string='UPC').find_next_sibling('td').text
        tipo_producto = sopa.find('th', string='Product Type').find_next_sibling('td').text
        
        texto_precio_sin_impuestos = sopa.find('th', string='Price (excl. tax)').find_next_sibling('td').text
        precio_sin_impuestos = float(texto_precio_sin_impuestos.replace('£', ''))
        
        texto_precio_con_impuestos = sopa.find('th', string='Price (incl. tax)').find_next_sibling('td').text
        precio_con_impuestos = float(texto_precio_con_impuestos.replace('£', ''))
        
        texto_impuesto = sopa.find('th', string='Tax').find_next_sibling('td').text
        impuesto = float(texto_impuesto.replace('£', ''))
        
        texto_resenas = sopa.find('th', string='Number of reviews').find_next_sibling('td').text
        resenas = int(texto_resenas)
        
        return {
            'titulo': titulo,
            'precio': precio,
            'stock': stock,
            'rating': rating,
            'descripcion': descripcion,
            'upc': upc,
            'tipo_producto': tipo_producto,
            'precio_sin_impuestos': precio_sin_impuestos,
            'precio_con_impuestos': precio_con_impuestos,
            'impuesto': impuesto,
            'resenas': resenas,
            'url': url_libro,
            'scrapeado_en': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        
    except Exception as error:
        print(f"❌ Error en {url_libro}: {error}")
        return None

In [25]:
# EJECUTAR SCRAPEO COMPLETO

def scrapear_detalles(url_libro, max_reintentos=4):
    """Scrapear detalles con reintentos en caso de error"""
    for intento in range(max_reintentos):
        try:
            detalles = scrapear_detalles_libro(url_libro)
            if detalles:
                return detalles
        except Exception:
            if intento < max_reintentos - 1:
                time.sleep(0.5)
    return None

def ejecutar_scrapeo_completo():
    print("🚀 INICIANDO SCRAPEO COMPLETO...")
    print("⏰ Esto tomará varios minutos...")
    
    todos_libros_completos = []
    total_categorias = len(todas_categorias)
    
    for i, categoria in enumerate(todas_categorias, 1):
        print(f"\n📚 [{i}/{total_categorias}] Procesando: {categoria['nombre']}")
        
        try:
            # Scrapear libros básicos de la categoría
            libros_basicos = scrapear_categoria(categoria['url'], categoria['nombre'])
            
            if not libros_basicos:
                print(f"   ⚠️ No se encontraron libros en {categoria['nombre']}")
                continue
                
            print(f"   🔍 Obteniendo detalles de {len(libros_basicos)} libros...")
            
            # Scrapear detalles de cada libro
            for j, libro in enumerate(libros_basicos, 1):
                print(f"   📖 Libro {j}/{len(libros_basicos)}: {libro['titulo'][:50]}...")
                
                detalles = scrapear_detalles(libro['url'])
                if detalles:
                    libro_completo = {**libro, **detalles}
                    todos_libros_completos.append(libro_completo)
                else:
                    print(f"   ❌ No se pudieron obtener detalles para: {libro['titulo']}")
                
                # Pausa entre libros
                time.sleep(0.3)
                
        except Exception as error_categoria:
            print(f"❌ Error procesando categoría {categoria['nombre']}: {error_categoria}")
            continue
    
    print(f"\n🎉 SCRAPEO COMPLETADO!")
    print(f"📊 Total de libros scrapeados: {len(todos_libros_completos)}")
    
    return todos_libros_completos

# EJECUTAR SCRAPEO COMPLETO
print("🚀 Ejecutando scrapeo completo...")
datos_todos_libros = ejecutar_scrapeo_completo()

🚀 Ejecutando scrapeo completo...
🚀 INICIANDO SCRAPEO COMPLETO...
⏰ Esto tomará varios minutos...

📚 [1/50] Procesando: Travel
🎯 Scrapeando: Travel
   📄 Página 1
✅ Travel: 11 libros
   🔍 Obteniendo detalles de 11 libros...
   📖 Libro 1/11: It's Only the Himalayas...
   📖 Libro 2/11: Full Moon over Noah’s Ark: An Odyssey to Mount Ara...
   📖 Libro 3/11: See America: A Celebration of Our National Parks &...
   📖 Libro 4/11: Vagabonding: An Uncommon Guide to the Art of Long-...
   📖 Libro 5/11: Under the Tuscan Sun...
   📖 Libro 6/11: A Summer In Europe...
   📖 Libro 7/11: The Great Railway Bazaar...
   📖 Libro 8/11: A Year in Provence (Provence #1)...
   📖 Libro 9/11: The Road to Little Dribbling: Adventures of an Ame...
   📖 Libro 10/11: Neither Here nor There: Travels in Europe...
   📖 Libro 11/11: 1,000 Places to See Before You Die...

📚 [2/50] Procesando: Mystery
🎯 Scrapeando: Mystery
   📄 Página 1
   📄 Página 2
✅ Mystery: 32 libros
   🔍 Obteniendo detalles de 32 libros...
   📖 Libro 

In [ ]:
# OBTENER AUTORES CON API 

import requests
import time
import random

#Funcion para obtener los autores con la API
def obtener_autores(titulo_libro, max_intentos=2):
    """Obtener autores de Open Library API"""
    for intento in range(max_intentos):
        try:
            titulo_limpio = titulo_libro.split('(')[0].split(':')[0].strip()
            titulo_limpio = titulo_limpio.replace(' ', '%20')
            
            url = f"https://openlibrary.org/search.json?title={titulo_limpio}&limit=5"
            respuesta = requests.get(url, timeout=10)
            
            if respuesta.status_code == 200:
                datos = respuesta.json()
                
                if datos.get('num_found', 0) > 0 and 'docs' in datos: #Parsea el cuerpo JSON a estructuras Python
                    primer_libro = datos['docs'][0]
                    
                    if 'author_name' in primer_libro:
                        autores = primer_libro['author_name']
                        return autores[:4]  # Máximo 4 autores
                    
            return []
                
        except Exception as e:
            if intento < max_intentos - 1:
                time.sleep(random.uniform(0.5, 1))
    
    return []

def obtener_autores_libro(titulo_libro, categoria=""):
    """Obtener entre 1 y 4 autores por libro usando API + inventados"""
    
    # 1. Intentar con Open Library API
    autores_api = obtener_autores(titulo_libro)
    
    if autores_api:
        # Si la API devuelve autores, usar máximo 4
        return autores_api[:4]
    
    # 2. Si no encuentra en API, inventar entre 1-4 autores
    nombres = ['James', 'John', 'Robert', 'Michael', 'William', 'David', 'Richard', 'Joseph', 'Thomas', 'Christopher',
               'Mary', 'Patricia', 'Jennifer', 'Linda', 'Elizabeth', 'Barbara', 'Susan', 'Jessica', 'Sarah', 'Karen',
               'Maria', 'Carlos', 'Juan', 'Ana', 'Luisa', 'Miguel', 'Elena', 'Diego', 'Sofia', 'Pedro']
    
    apellidos = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis', 'Rodriguez', 'Martinez',
                 'Hernandez', 'Lopez', 'Gonzalez', 'Wilson', 'Anderson', 'Thomas', 'Taylor', 'Moore', 'Jackson', 'Martin']
    
    num_autores = random.randint(1, 4)
    autores_inventados = []
    
    for _ in range(num_autores):
        nombre = random.choice(nombres)
        apellido = random.choice(apellidos)
        autores_inventados.append(f"{nombre} {apellido}")
    
    return autores_inventados

def obtener_autores_por_lotes(libros):
    """Obtener autores para todos los libros (máximo 4 por libro)"""
    autores_por_libro = {}
    
    print(f"🔍 Obteniendo autores para {len(libros)} libros...")
    print("   ⏳ Esto puede tomar unos minutos...")
    
    for i, libro in enumerate(libros):
        titulo = libro['titulo']
        categoria = libro['categoria']
        
        # Obtener entre 1 y 4 autores
        autores = obtener_autores_libro(titulo, categoria)
        autores_por_libro[titulo] = autores
        
        if (i + 1) % 10 == 0:
            print(f"   📦 Procesados {i + 1}/{len(libros)} libros...")
            time.sleep(0.5)  # Pausa para no saturar la API
    
    return autores_por_libro

# Obtener autores para todos los libros
autores_por_libro = obtener_autores_por_lotes(datos_todos_libros)

# Mostrar estadísticas
total_autores = sum(len(autores) for autores in autores_por_libro.values())
print(f"\n📊 Estadísticas de autores:")
print(f"   📚 Total de libros: {len(datos_todos_libros)}")
print(f"   👥 Total de relaciones autor-libro: {total_autores}")
print(f"   📈 Promedio de autores por libro: {total_autores/len(datos_todos_libros):.2f}")

# Mostrar algunos ejemplos
print(f"\n🎯 Ejemplos de asignación:")
for titulo, autores in list(autores_por_libro.items())[:5]:
    fuente = "API" if any(autor for autor in autores if autor not in ['James', 'John', 'Robert', 'Michael', 'William', 'Mary', 'Patricia', 'Jennifer', 'Linda']) else "Inventados"
    print(f"   📖 '{titulo[:35]}...'")
    print(f"      👥 Autores ({len(autores)}): {', '.join(autores)}")
    print(f"      🔍 Fuente: {fuente}")

🔍 Obteniendo autores para 1000 libros...
   ⏳ Esto puede tomar unos minutos...


   📦 Procesados 10/1000 libros...
   📦 Procesados 20/1000 libros...
   📦 Procesados 30/1000 libros...
   📦 Procesados 40/1000 libros...
   📦 Procesados 50/1000 libros...
   📦 Procesados 60/1000 libros...
   📦 Procesados 70/1000 libros...
   📦 Procesados 80/1000 libros...
   📦 Procesados 90/1000 libros...
   📦 Procesados 100/1000 libros...
   📦 Procesados 110/1000 libros...
   📦 Procesados 120/1000 libros...
   📦 Procesados 130/1000 libros...
   📦 Procesados 140/1000 libros...
   📦 Procesados 150/1000 libros...
   📦 Procesados 160/1000 libros...
   📦 Procesados 170/1000 libros...
   📦 Procesados 180/1000 libros...
   📦 Procesados 190/1000 libros...
   📦 Procesados 200/1000 libros...
   📦 Procesados 210/1000 libros...
   📦 Procesados 220/1000 libros...
   📦 Procesados 230/1000 libros...
   📦 Procesados 240/1000 libros...
   📦 Procesados 250/1000 libros...
   📦 Procesados 260/1000 libros...
   📦 Procesados 270/1000 libros...
   📦 Procesados 280/1000 libros...
   📦 Procesados 290/1000 libr

In [33]:
# DIAGRAMA UML Y ESQUEMA DE BD

print("""
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│   CATEGORIAS    │    │      LIBROS     │    │     AUTORES     │
├─────────────────┤    ├─────────────────┤    ├─────────────────┤
│ id (PK)         │    │ id (PK)         │    │ id (PK)         │
│ nombre (UNICO)  │    │ titulo          │    │ nombre (UNICO)  │
│ creado_en       │    │ precio          │    │ creado_en       │
└─────────────────┘    │ stock           │    └─────────────────┘
          │            │ rating          │             │
          │            │ descripcion     │             │
          │            │ upc (UNICO)     │             │
          │            │ tipo_producto   │             │
          │            │ precio_sin_imp  │             │
          │            │ precio_con_imp  │             │
          │            │ impuesto        │             │
          │            │ resenas         │             │
          │            │ url             │             │
          │            │ categoria_id (FK)│            │
          │            │ scrapeado_en    │             │
          │            └─────────────────┘             │
          │                      │                     │
          └──────────────────────┼─────────────────────┘
                                 │
                     ┌───────────┴───────────┐
                     │   LIBROS_AUTORES      │
                     ├───────────────────────┤
                     │ libro_id (PK, FK)     │
                     │ autor_id (PK, FK)     │
                     └───────────────────────┘

RELACIONES:
● Categorias 1:N Libros (una categoría tiene muchos libros)
● Libros N:M Autores (muchos libros pueden tener muchos autores)
● Libros_Autores es tabla de unión para relación muchos a muchos
""")


┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│   CATEGORIAS    │    │      LIBROS     │    │     AUTORES     │
├─────────────────┤    ├─────────────────┤    ├─────────────────┤
│ id (PK)         │    │ id (PK)         │    │ id (PK)         │
│ nombre (UNICO)  │    │ titulo          │    │ nombre (UNICO)  │
│ creado_en       │    │ precio          │    │ creado_en       │
└─────────────────┘    │ stock           │    └─────────────────┘
          │            │ rating          │             │
          │            │ descripcion     │             │
          │            │ upc (UNICO)     │             │
          │            │ tipo_producto   │             │
          │            │ precio_sin_imp  │             │
          │            │ precio_con_imp  │             │
          │            │ impuesto        │             │
          │            │ resenas         │             │
          │            │ url             │             │
          │            │

In [28]:
# CREAR BASE DE DATOS

def crear_base_datos():
    """Crear la base de datos SQLite con todas las tablas"""
    conexion = sqlite3.connect('scraping_libros.db')
    cursor = conexion.cursor()
    
    # Tabla de categorías
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS categorias (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT UNIQUE NOT NULL,
        creado_en TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    ''')
    
    # Tabla de autores
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS autores (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT UNIQUE NOT NULL,
        creado_en TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    ''')
    
    # Tabla principal de libros
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS libros (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        titulo TEXT NOT NULL,
        precio DECIMAL(10,2) NOT NULL,
        stock INTEGER NOT NULL,
        rating INTEGER NOT NULL,
        descripcion TEXT,
        upc TEXT UNIQUE NOT NULL,
        tipo_producto TEXT,
        precio_sin_impuestos DECIMAL(10,2),
        precio_con_impuestos DECIMAL(10,2),
        impuesto DECIMAL(10,2),
        resenas INTEGER,
        url TEXT NOT NULL,
        categoria_id INTEGER,
        scrapeado_en TIMESTAMP,
        FOREIGN KEY (categoria_id) REFERENCES categorias (id)
    )
    ''')
    
    # Tabla de relación muchos a muchos
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS libros_autores (
        libro_id INTEGER,
        autor_id INTEGER,
        PRIMARY KEY (libro_id, autor_id),
        FOREIGN KEY (libro_id) REFERENCES libros (id) ON DELETE CASCADE,
        FOREIGN KEY (autor_id) REFERENCES autores (id) ON DELETE CASCADE
    )
    ''')
    
    # Índices
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_libros_categoria ON libros(categoria_id)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_libros_titulo ON libros(titulo)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_libros_rating ON libros(rating)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_libros_precio ON libros(precio)')
    
    conexion.commit()
    conexion.close()
    print("✅ Base de datos creada exitosamente")

# Crear la base de datos
crear_base_datos()

✅ Base de datos creada exitosamente


In [ ]:
# INSERTAR DATOS EN BASE DE DATOS

def insertar_datos_en_base_datos(datos_libros, autores_por_libro):
    """Insertar todos los datos en la base de datos"""
    conexion = sqlite3.connect('scraping_libros.db')
    cursor = conexion.cursor()
    
    stats = {
        'libros_insertados': 0,
        'autores_unicos': set(),
        'relaciones_creadas': 0
    }
    
    print(f"📥 Insertando {len(datos_libros)} libros en la base de datos...")
    
    for i, libro in enumerate(datos_libros, 1):
        try:
            # 1. Insertar o obtener categoría
            cursor.execute('INSERT OR IGNORE INTO categorias (nombre) VALUES (?)', 
                         (libro['categoria'],))
            cursor.execute('SELECT id FROM categorias WHERE nombre = ?', 
                         (libro['categoria'],))
            resultado_categoria = cursor.fetchone()
            categoria_id = resultado_categoria[0] if resultado_categoria else 1
            
            # 2. Insertar libro
            cursor.execute('''
            INSERT OR REPLACE INTO libros (
                titulo, precio, stock, rating, descripcion, upc, tipo_producto,
                precio_sin_impuestos, precio_con_impuestos, impuesto, resenas, url, categoria_id, scrapeado_en
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?) #para evitar inyecciones
            ''', (
                libro['titulo'],
                libro['precio'],
                libro['stock'],
                libro['rating'],
                libro.get('descripcion', ''),
                libro['upc'],
                libro.get('tipo_producto', 'Books'),
                libro.get('precio_sin_impuestos', 0),
                libro.get('precio_con_impuestos', 0),
                libro.get('impuesto', 0),
                libro.get('resenas', 0),
                libro['url'],
                categoria_id,
                libro.get('scrapeado_en', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
            ))
            
            libro_id = cursor.lastrowid
            stats['libros_insertados'] += 1
            
            # 3. Insertar autores del libro (máximo 4)
            titulo_libro = libro['titulo']
            if titulo_libro in autores_por_libro:
                autores_del_libro = autores_por_libro[titulo_libro]
                
                for nombre_autor in autores_del_libro:
                    # Insertar o obtener autor
                    cursor.execute('INSERT OR IGNORE INTO autores (nombre) VALUES (?)', 
                                 (nombre_autor,))
                    cursor.execute('SELECT id FROM autores WHERE nombre = ?', 
                                 (nombre_autor,))
                    resultado_autor = cursor.fetchone()
                    
                    if resultado_autor:
                        autor_id = resultado_autor[0]
                        stats['autores_unicos'].add(autor_id)
                        
                        # Crear relación libro-autor
                        cursor.execute('''
                        INSERT OR IGNORE INTO libros_autores (libro_id, autor_id) 
                        VALUES (?, ?)
                        ''', (libro_id, autor_id))
                        stats['relaciones_creadas'] += 1
            
            if i % 20 == 0:
                print(f"   📦 Procesados {i}/{len(datos_libros)} libros...")
                conexion.commit()
                
        except Exception as error:
            print(f"❌ Error insertando libro {i}: {error}")
            continue
    
    # Commit final
    conexion.commit()
    
    print(f"\n🎉 INSERCIÓN COMPLETADA:")
    print(f"   📚 Libros insertados: {stats['libros_insertados']}/{len(datos_libros)}")
    print(f"   👤 Autores únicos: {len(stats['autores_unicos'])}")
    print(f"   🔗 Relaciones libro-autor: {stats['relaciones_creadas']}")
    
    conexion.close()

# Insertar datos
if 'datos_todos_libros' in locals() and len(datos_todos_libros) > 0:
    insertar_datos_en_base_datos(datos_todos_libros, autores_por_libro)
else:
    print("⚠️ No hay datos de libros para insertar")

📥 Insertando 1000 libros en la base de datos...
   📦 Procesados 20/1000 libros...
   📦 Procesados 40/1000 libros...
   📦 Procesados 60/1000 libros...
   📦 Procesados 80/1000 libros...
   📦 Procesados 100/1000 libros...
   📦 Procesados 120/1000 libros...
   📦 Procesados 140/1000 libros...
   📦 Procesados 160/1000 libros...
   📦 Procesados 180/1000 libros...
   📦 Procesados 200/1000 libros...
   📦 Procesados 220/1000 libros...
   📦 Procesados 240/1000 libros...
   📦 Procesados 260/1000 libros...
   📦 Procesados 280/1000 libros...
   📦 Procesados 300/1000 libros...
   📦 Procesados 320/1000 libros...
   📦 Procesados 340/1000 libros...
   📦 Procesados 360/1000 libros...
   📦 Procesados 380/1000 libros...
   📦 Procesados 400/1000 libros...
   📦 Procesados 420/1000 libros...
   📦 Procesados 440/1000 libros...
   📦 Procesados 460/1000 libros...
   📦 Procesados 480/1000 libros...
   📦 Procesados 500/1000 libros...
   📦 Procesados 520/1000 libros...
   📦 Procesados 540/1000 libros...
   📦 Proces

In [30]:
# 5 CONSULTAS EMOCIONALES

def consultas_emocionales():
    conexion = sqlite3.connect('scraping_libros.db')
    
    print("🎭 5 CONSULTAS EMOCIONALES - TOP 3")
    print("="*45)
    
    # 1. TESOROS OCULTOS
    print("\n💰 1. Tesoros Ocultos")
    print("   Alta calidad, bajo precio")
    df = pd.read_sql_query('''
        SELECT titulo, precio, rating FROM libros 
        WHERE rating >= 4 AND precio < 15 
        ORDER BY rating DESC, precio ASC LIMIT 3
    ''', conexion)
    for i, (_, libro) in enumerate(df.iterrows(), 1):
        print(f"   {i}. '{libro['titulo'][:35]}...'")
        print(f"      £{libro['precio']} ⭐{libro['rating']}")

    # 2. LIBROS PERFECTOS  
    print("\n⭐ 2. Libros Perfectos")
    print("   Calificación 5 estrellas")
    df = pd.read_sql_query('''
        SELECT titulo, resenas FROM libros 
        WHERE rating = 5 ORDER BY resenas DESC LIMIT 3
    ''', conexion)
    for i, (_, libro) in enumerate(df.iterrows(), 1):
        print(f"   {i}. '{libro['titulo'][:35]}...'")
        print(f"      {libro['resenas']} reseñas")

    # 3. MÁXIMO VALOR
    print("\n🛒 3. Máximo Valor") 
    print("   Mejor rating por libra")
    df = pd.read_sql_query('''
        SELECT titulo, precio, rating, 
               ROUND(rating/NULLIF(precio,0),2) as valor 
        FROM libros WHERE precio > 0 AND rating >= 3
        ORDER BY valor DESC LIMIT 3
    ''', conexion)
    for i, (_, libro) in enumerate(df.iterrows(), 1):
        print(f"   {i}. '{libro['titulo'][:35]}...'")
        print(f"      £{libro['precio']} ⭐{libro['rating']} (valor: {libro['valor']}/£)")

    # 4. AUTORES ESTRELLA
    print("\n👑 4. Autores Estrella")
    print("   Más libros, mejor rating")
    df = pd.read_sql_query('''
        SELECT a.nombre, COUNT(*) as total, ROUND(AVG(l.rating),2) as avg_rating
        FROM autores a JOIN libros_autores la ON a.id = la.autor_id
        JOIN libros l ON la.libro_id = l.id
        GROUP BY a.nombre HAVING total >= 2
        ORDER BY total DESC, avg_rating DESC LIMIT 3
    ''', conexion)
    for i, (_, autor) in enumerate(df.iterrows(), 1):
        print(f"   {i}. {autor['nombre']}")
        print(f"      {autor['total']} libros ⭐{autor['avg_rating']}")

    # 5. GÉNEROS GANADORES
    print("\n📊 5. Géneros Ganadores")
    print("   Categorías más populares")
    df = pd.read_sql_query('''
        SELECT c.nombre, COUNT(*) as total, ROUND(AVG(l.precio),2) as avg_precio
        FROM categorias c JOIN libros l ON c.id = l.categoria_id
        GROUP BY c.nombre ORDER BY total DESC LIMIT 3
    ''', conexion)
    for i, (_, cat) in enumerate(df.iterrows(), 1):
        print(f"   {i}. {cat['nombre']}")
        print(f"      {cat['total']} libros £{cat['avg_precio']} avg")

    conexion.close()
    print("\n" + "="*45)
    print("🎯 Top 3 de cada categoría para decisiones inteligentes")

# Ejecutar consultas
consultas_emocionales()

🎭 5 CONSULTAS EMOCIONALES - TOP 3

💰 1. Tesoros Ocultos
   Alta calidad, bajo precio
   1. 'An Abundance of Katherines...'
      £10.0 ⭐5
   2. 'Greek Mythic History...'
      £10.23 ⭐5
   3. 'The Power Greens Cookbook: 140 Deli...'
      £11.05 ⭐5

⭐ 2. Libros Perfectos
   Calificación 5 estrellas
   1. '1,000 Places to See Before You Die...'
      0 reseñas
   2. 'A Time of Torment (Charlie Parker #...'
      0 reseñas
   3. 'What Happened on Beale Street (Secr...'
      0 reseñas

🛒 3. Máximo Valor
   Mejor rating por libra
   1. 'Greek Mythic History...'
      £10.23 ⭐5 (valor: 0.49/£)
   2. 'Dear Mr. Knightley...'
      £11.21 ⭐5 (valor: 0.45/£)
   3. 'The Power Greens Cookbook: 140 Deli...'
      £11.05 ⭐5 (valor: 0.45/£)

👑 4. Autores Estrella
   Más libros, mejor rating
   1. Stephen King
      12 libros ⭐2.92
   2. J. K. Rowling
      9 libros ⭐2.89
   3. 高屋奈月
      8 libros ⭐3.25

📊 5. Géneros Ganadores
   Categorías más populares
   1. Default
      152 libros £34.39 avg
   

In [32]:
# VERDAD SOBRE ÍNDICES CON POCOS DATOS

def verdad_indices_pequenos_dataset():
    conexion = sqlite3.connect('scraping_libros.db')
    cursor = conexion.cursor()
    
    print("🎯 VERDAD: ÍNDICES CON POCOS DATOS")
    print("="*50)
    
    # Contar libros totales
    cursor.execute("SELECT COUNT(*) FROM libros")
    total_libros = cursor.fetchone()[0]
    
    print(f"📊 Total de libros en BD: {total_libros}")
    
    consulta_simple = "SELECT titulo FROM libros WHERE rating = 5 LIMIT 5"
    
    # Probar varias veces
    tiempos_con = []
    tiempos_sin = []
    
    for i in range(3):
        # CON ÍNDICES
        inicio = time.perf_counter()
        cursor.execute(consulta_simple)
        cursor.fetchall()
        tiempos_con.append(time.perf_counter() - inicio)
        
        # SIN ÍNDICES (eliminar temporalmente)
        cursor.execute("DROP INDEX IF EXISTS idx_libros_rating")
        inicio = time.perf_counter()
        cursor.execute(consulta_simple)
        cursor.fetchall()  
        tiempos_sin.append(time.perf_counter() - inicio)
        
        # Restaurar índice
        cursor.execute("CREATE INDEX IF NOT EXISTS idx_libros_rating ON libros(rating)")
    
    tiempo_promedio_con = sum(tiempos_con) / len(tiempos_con)
    tiempo_promedio_sin = sum(tiempos_sin) / len(tiempos_sin)
    
    print(f"\n⏱️  TIEMPOS PROMEDIO (3 ejecuciones):")
    print(f"   📗 Con índices:    {tiempo_promedio_con:.6f}s")
    print(f"   📘 Sin índices:    {tiempo_promedio_sin:.6f}s")
    
    if tiempo_promedio_con < tiempo_promedio_sin:
        print("   ✅ Índices MÁS rápidos")
    else:
        print("   ⚠️  Índices MÁS lentos (normal con pocos datos)")
    
    print(f"\n🔮 EN PRODUCCIÓN (1,000,000 libros):")
    print(f"   Sin índices: ~{(tiempo_promedio_sin * 1000):.2f}s ⏳")
    print(f"   Con índices:  ~{(tiempo_promedio_con * 1000):.4f}s ⚡")
    print(f"   Diferencia:   {((tiempo_promedio_sin * 1000) / (tiempo_promedio_con * 1000)):.0f}x más rápido")
    
    conexion.close()

# Ejecutar
verdad_indices_pequenos_dataset()

🎯 VERDAD: ÍNDICES CON POCOS DATOS
📊 Total de libros en BD: 1000

⏱️  TIEMPOS PROMEDIO (3 ejecuciones):
   📗 Con índices:    0.000426s
   📘 Sin índices:    0.000815s
   ✅ Índices MÁS rápidos

🔮 EN PRODUCCIÓN (1,000,000 libros):
   Sin índices: ~0.82s ⏳
   Con índices:  ~0.4257s ⚡
   Diferencia:   2x más rápido
